# 06 Mga signal, threshold at 7-1-7 — notebook

Agosto sa Santa Luz: humuhupa na ang alon ng leptospirosis, pero hindi ang naiwang tubig, at abala ang mga lamok na *Aedes*. Simple ang tanong ni Maya tuwing Lunes: **hindi ba pangkaraniwan ang bilang ng dengue ngayong linggo, o tag-ulan lang talaga?** Sa notebook na ito, gagawin mo ang mga sumusunod:

1. basahin ang lingguhang series ng dengue (`data/synthetic/ph/santa_luz_dengue_weekly.csv`, **synthetic**);
2. bumuo ng **epidemic threshold** — mean + 2 SD ng parehong mga linggo sa karaniwang mga taon — at hanapin ang unang linggo ng 2026 na lumampas dito;
3. gumuhit ng **endemic channel**;
4. alamin kung bakit laging mukhang mababa ang pinakabagong mga linggo (mga huling ulat);
5. sukatin ang **7-1-7 timeliness** sa 40 lumipas na event (`signals_717.csv`) at basahin kung ano ang nagpabagal sa mga ito.

Gawa-gawa ang bawat bilang dito; totoo ang mga paraan.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Hakbang 1 — Ang lingguhang series

Isang row bawat **morbidity week**. ISO week ang gamit ng file na ito (Lunes hanggang Linggo, may label na `2026-W32`); magkahiwalay na column ang `iso_year` at `iso_week` para mapagtabi natin ang "parehong linggo" sa iba't ibang taon. I-load natin ito at tingnan ang 2026 sa ngayon.

In [ ]:
import statistics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

weekly = pd.read_csv("data/synthetic/ph/santa_luz_dengue_weekly.csv", parse_dates=["week_start"])
print(weekly.shape, "| mga taon:", sorted(weekly["iso_year"].unique()))
print("Linggo bawat taon:", weekly.groupby("iso_year").size().to_dict())

y2026 = weekly[weekly["iso_year"] == 2026].set_index("iso_week")
print("\n2026, linggo 28-39:")
print(y2026.loc[28:39, ["morbidity_week", "week_start", "cases"]].to_string())

## Hakbang 2 — Buuin ang threshold

Ang patakarang ginagamit ng CESU para sa series na ito (nakasulat ito sa README ng datos):

* para sa ISO week *w*, kunin ang bilang ng mga linggong *w*−1, *w* at *w*+1 sa **baseline years 2022–2025** — 4 na taon × 3 linggo = **12 numero**;
* ang threshold ay ang **mean + 2 standard deviation** nila (sample SD);
* hindi isinama ang 2019 dahil epidemic year ito, at ang 2020–2021 dahil ginulo ng COVID-19 ang pagpapakonsulta at pag-uulat. Dapat *karaniwang* taon ang inilalarawan ng baseline.

> **Tanungin si Tala:** *May pandas DataFrame akong `weekly` na may mga column na `iso_year`, `iso_week` (1–53) at `cases`. Mga gawa-gawang row: `(2022, 31, 9)`, `(2022, 32, 12)`, `(2023, 32, 7)`. Para sa bawat ISO week w mula 1 hanggang 53, kuwentahin ang epidemic threshold = mean + 2 × sample SD ng `cases` sa mga linggong w−1, w, w+1 ng mga taong 2022–2025 lang (ang window ng week 1 ay iikot pabalik sa week 52). Ibalik ito bilang Series na naka-index ayon sa linggo. Tapos sabihin mo kung aling linggo ng 2026 ang unang lumampas sa threshold nito. Pakigawang madaling basahin para sa baguhan.*

In [ ]:
BASELINE_YEARS = [2022, 2023, 2024, 2025]   # karaniwang taon lang: hindi 2019, 2020, 2021, at hinding-hindi 2026


def window_weeks(week):
    # Linggong w-1, w, w+1; ang week 1 ay iikot sa 52 at ang week 52 sa 1 (ang week 53 ay 52, 53, 1)
    if week == 53:
        return {52, 53, 1}
    return {(week - 2) % 52 + 1, week, week % 52 + 1}


def epidemic_threshold(weekly, years):
    base = weekly[weekly["iso_year"].isin(years)]
    rows = {}
    for week in range(1, 54):
        values = base.loc[base["iso_week"].isin(window_weeks(week)), "cases"]
        rows[week] = {"n_values": len(values), "baseline_mean": values.mean(),
                      "threshold": values.mean() + 2 * values.std(ddof=1)}
    return pd.DataFrame(rows).T


thr = epidemic_threshold(weekly, BASELINE_YEARS)
season = y2026[["cases"]].join(thr)
season["above"] = season["cases"] > season["threshold"]
print(season.loc[28:39].round(1).to_string())

first_week = season.index[season["above"]].min()
print(f"\nUnang linggo ng 2026 na lampas sa threshold: W{first_week} "
      f"({y2026.loc[first_week, 'week_start']:%d %b}): {season.loc[first_week, 'cases']:.0f} kaso "
      f"laban sa threshold na {season.loc[first_week, 'threshold']:.1f}")

**Suriin.** Tatlong check na gagawin ni Doc Mon:

1. **Kuwentahin nang mano-mano ang isang linggo.** Ilista ang 12 baseline value para sa W32 at kuwentahin ang mean + 2 SD gamit ang `statistics` module ng Python — ikalawang paraan na hiwalay sa pandas.
2. **Ang denominator ng window.** Dapat eksaktong 12 value ang bawat linggo.
3. **Ang kuwento.** Walang linggo ng 2026 bago ang unang paglampas na lampas sa threshold, at lampas ang bawat linggo pagkatapos nito.

In [ ]:
base = weekly[weekly["iso_year"].isin(BASELINE_YEARS)]
w32_values = base.loc[base["iso_week"].isin([31, 32, 33]), "cases"].tolist()
print("Ang 12 baseline value para sa W32:", w32_values)
by_hand = statistics.mean(w32_values) + 2 * statistics.stdev(w32_values)
print(f"mean + 2 SD, mano-mano: {by_hand:.2f}  | mula sa function: {thr.loc[32, 'threshold']:.2f}")

assert len(w32_values) == 12
assert abs(by_hand - thr.loc[32, "threshold"]) < 1e-9
assert (thr.loc[1:52, "n_values"] == 12).all()
assert not season.loc[:first_week - 1, "above"].any(), "walang naunang linggong puwedeng lampas"
assert season.loc[first_week:39, "above"].all()
print("Pasado lahat ng check: tutunog sana ang alert sa W32, at mananatiling bukas hanggang W39.")

## Hakbang 3 — Endemic channel

Iginuguhit ng **endemic channel** kung ano ang "karaniwan" sa bawat linggo ng taon, para mabasa ang curve ngayong taon laban dito. Iginuguhit natin ang bawat baseline year bilang manipis na kulay-abong linya, ang baseline mean (inaasahan) at ang threshold, tapos ang 2026 sa ibabaw.

In [ ]:
weeks = np.arange(1, 53)
fig, ax = plt.subplots(figsize=(11, 4.2))
for year in BASELINE_YEARS:
    one = weekly[weekly["iso_year"] == year].set_index("iso_week")["cases"]
    ax.plot(one.index, one.values, color="#aeb7c0", linewidth=0.8)
ax.plot(weeks, thr.loc[weeks, "baseline_mean"], color="#2f7d32", linewidth=2, label="Baseline mean (2022-2025)")
ax.fill_between(weeks, 0, thr.loc[weeks, "threshold"], color="#2f7d32", alpha=0.10)
ax.plot(weeks, thr.loc[weeks, "threshold"], color="#b45309", linewidth=2, linestyle="--",
        label="Epidemic threshold (mean + 2 SD)")
bar_colours = np.where(season["above"], "#CE1126", "#0038A8")
ax.bar(season.index, season["cases"], color=bar_colours, alpha=0.85, label="2026 (pula = lampas sa threshold)")
ax.annotate("Bagyong Amihan\n(W29)", xy=(29, 6), xytext=(18, 45), arrowprops={"arrowstyle": "->"})
ax.set_xlabel("ISO week")
ax.set_ylabel("Kaso ng dengue bawat linggo")
ax.set_title("Santa Luz City (synthetic): dengue ng 2026 laban sa endemic channel nito")
ax.set_xlim(0.5, 52.5)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Nagkamali si Tala: ang epidemyang nagtaas ng sarili nitong threshold

Sabi sa unang prompt ni Maya, "gamitin ang lahat ng taon sa file". Iyon nga mismo ang ginawa ni Tala — kaya kasama sa threshold ang **2026 mismo** (at ang 2019, isang epidemic year). Kapag nasa loob ng sarili nitong baseline ang isang epidemya, itinutulak nito pataas ang mean at ang SD, kaya sumasabay sa pagtaas ng curve ang threshold. Ipinapakita ng cell sa ibaba ang epekto, at ang isang-linyang `assert` na makakahuli sana rito.

In [ ]:
all_years = sorted(weekly["iso_year"].unique())          # bersiyon ni Tala: 2019 ... 2026
thr_wrong = epidemic_threshold(weekly, all_years)
compare = y2026.loc[30:39, ["cases"]].copy()
compare["right_threshold"] = thr.loc[30:39, "threshold"]
compare["wrong_threshold"] = thr_wrong.loc[30:39, "threshold"]
compare["alarm_right"] = compare["cases"] > compare["right_threshold"]
compare["alarm_wrong"] = compare["cases"] > compare["wrong_threshold"]
print(compare.round(1).to_string())
print(f"\nMga linggong W32-W39 na na-flag: tamang baseline {int(compare.loc[32:, 'alarm_right'].sum())} sa 8, "
      f"baseline ni Tala {int(compare.loc[32:, 'alarm_wrong'].sum())} sa 8")
silenced = compare.index[compare["alarm_right"] & ~compare["alarm_wrong"]].tolist()
print(f"Tumaas ang threshold ng W32 mula {thr.loc[32, 'threshold']:.1f} papuntang {thr_wrong.loc[32, 'threshold']:.1f}; "
      f"tahimik ang alarma sa mga linggong {silenced} habang umaakyat ang epidemya")

# Ang bantay na nakakahuli nito: hindi puwedeng nasa sarili nitong baseline ang season na sinusuri mo
def check_baseline(years, season_year=2026, epidemic_years=(2019,)):
    assert season_year not in years, f"nasa sarili nitong baseline ang {season_year}"
    assert not set(epidemic_years) & set(years), "may epidemic year sa baseline"

check_baseline(BASELINE_YEARS)            # tahimik na pumapasa
try:
    check_baseline(all_years)
except AssertionError as err:
    print("Nahuli:", err)

## Hakbang 4 — Bakit laging mukhang mababa ang pinakabagong mga linggo

Binibilang ng lingguhang file ang mga kaso ayon sa **linggo ng onset**, pero ilang araw pa bago makarating ang kaso sa CESU (konsulta, tapos notification). Kaya sa anumang Miyerkoles, hindi pa kumpleto ang katatapos na linggo, at halos walang laman ang kasalukuyang linggo. Mauulit natin ito gamit ang line list mula sa [kabanata 04](../04_time_place_person.md): bilangin ang mga kaso ng dengue ayon sa linggo ng onset gamit lang ang mga row na **naiulat hanggang** sa isang petsa.

In [ ]:
ll = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv", parse_dates=["date_onset", "date_report"])
dengue = ll[ll["disease"] == "dengue"].copy()
dengue["onset_week"] = dengue["date_onset"].dt.isocalendar().week.astype(int)
delay_days = (dengue["date_report"] - dengue["date_onset"]).dt.days
print(f"Dengue, onset -> ulat: median {delay_days.median():.0f} araw, 90% sa loob ng {delay_days.quantile(0.9):.0f} araw")

snapshots = {}
for as_of in ["2026-09-02", "2026-09-16", "2026-09-30"]:
    known = dengue[dengue["date_report"] <= as_of]
    snapshots[f"alam noong {as_of[5:]}"] = known.groupby("onset_week").size()
snap = pd.DataFrame(snapshots).reindex(range(30, 40)).fillna(0).astype(int)
snap.index = [f"W{w}" for w in snap.index]
print(snap.T.to_string())

ax = snap.plot(kind="bar", rot=0, figsize=(10, 3.8), color=["#aeb7c0", "#FF9F1C", "#0038A8"], width=0.8)
ax.set_xlabel("Linggo ng onset (2026)")
ax.set_ylabel("Kaso ng dengue")
ax.set_title("Parehong mga linggo, tiningnan sa tatlong magkaibang araw")
plt.tight_layout()
plt.show()

## Hakbang 5 — 7-1-7: gaano kabilis ang response?

Ang **7-1-7** ay timeliness target para sa bawat public-health event:

* **7** — ma-detect ito sa loob ng **7 araw** mula emergence (onset ng unang kaso o simula ng event);
* **1** — i-notify ang public-health authority (dito, ang PESU) sa loob ng **1 araw** mula detection;
* **7** — tapusin ang early response sa loob ng **7 araw** mula notification.

Nasa `signals_717.csv` ang 40 synthetic na event mula Enero 2025 hanggang Agosto 2026, bawat isa may apat na petsa at isang column na `bottleneck` na nagsasabi kung ano ang dahilan ng bawat target na hindi naabot.

> **Tanungin si Tala:** *May DataFrame akong `events` na may mga date column na `date_emergence`, `date_detection`, `date_notification`, `date_response_complete` (gawa-gawang row: 2025-03-01, 2025-03-05, 2025-03-06, 2025-03-12). Magdagdag ng tatlong column ng mga delay sa buong araw (detection − emergence, notification − detection, response − notification), tatlong True/False na column para sa mga 7-1-7 target (≤ 7, ≤ 1, ≤ 7) at isa para sa "pasado sa tatlo". Tapos i-print kung ilang event ang pumasa sa bawat target, bilang count at porsiyento.*

In [ ]:
events = pd.read_csv(
    "data/synthetic/ph/signals_717.csv",
    parse_dates=["date_emergence", "date_detection", "date_notification", "date_response_complete"],
)
events["days_to_detect"] = (events["date_detection"] - events["date_emergence"]).dt.days
events["days_to_notify"] = (events["date_notification"] - events["date_detection"]).dt.days
events["days_to_respond"] = (events["date_response_complete"] - events["date_notification"]).dt.days

events["detect_ok"] = events["days_to_detect"] <= 7
events["notify_ok"] = events["days_to_notify"] <= 1
events["respond_ok"] = events["days_to_respond"] <= 7
events["all_ok"] = events[["detect_ok", "notify_ok", "respond_ok"]].all(axis=1)

targets = ["detect_ok", "notify_ok", "respond_ok", "all_ok"]
summary = pd.DataFrame({
    "met": events[targets].sum(),
    "missed": (~events[targets]).sum(),
    "percent_met": (events[targets].mean() * 100).round(0),
})
print(summary.to_string())
print("\nMedian na delay (araw):", events[["days_to_detect", "days_to_notify", "days_to_respond"]].median().to_dict())

**Suriin.** Bilangin nang mano-mano ang isang event. Ang EV-040 ay ang dengue surge sa Barangay Baybay sa kuwento: emergence noong Lunes ng linggong lumampas sa threshold. Basahin ang apat na petsa nito, ibawas ang mga ito sa papel, at ikumpara sa mga column ni Tala. Tapos tiyaking hindi kailanman puwedeng mas malaki ang "pasado sa tatlo" kaysa sa alinmang iisang target.

In [ ]:
ev = events.set_index("event_id").loc["EV-040"]
print(ev[["disease", "barangay", "date_emergence", "date_detection", "date_notification", "date_response_complete"]])
print("Mga delay:", ev["days_to_detect"], ev["days_to_notify"], ev["days_to_respond"])

assert ev["days_to_detect"] == (pd.Timestamp("2026-08-10") - pd.Timestamp("2026-08-03")).days == 7
assert ev["detect_ok"] and ev["notify_ok"] and not ev["respond_ok"]
assert summary.loc["all_ok", "met"] <= summary.loc[["detect_ok", "notify_ok", "respond_ok"], "met"].min()
assert (events[["days_to_detect", "days_to_notify", "days_to_respond"]] >= 0).all().all(), "negatibong delay = maling na-type na petsa"
print("Pasado ang mga check.")

## Hakbang 6 — Basahin ang column na bottleneck

Sinasabi ng porsiyento kung **gaano kadalas** hindi naabot ang isang target; sinasabi ng column na `bottleneck` kung **bakit**. Puwedeng may ilang dahilan ang isang event na pinagdugtong ng `"; "`, kaya hinahati natin ang mga ito, binibilang, at tinitingnan kung aling target ang hindi naabot ng bawat event. Ito ang listahang dadalhin ni Maya sa Local Health Board: mga problemang maaayos ng pondo, isang back-up na DSO o isang memo.

In [ ]:
missed = events[events["bottleneck"] != "none"]
causes = missed["bottleneck"].str.split("; ").explode().str.strip()
print(f"{len(missed)} event ang hindi umabot sa kahit isang target; {len(causes)} dahilan ang naitala.\n")
print(causes.value_counts().head(6).to_string())

print("\nHindi naabot bawat target:", (~events[["detect_ok", "notify_ok", "respond_ok"]]).sum().to_dict())
print("\nAng dalawang event sa Santa Luz noong 2026:")
print(events.tail(2)[["event_id", "disease", "barangay", "days_to_detect", "days_to_notify",
                      "days_to_respond", "bottleneck"]].to_string(index=False))

## Ikaw naman

1. **Mas mahigpit na threshold.** Kuwentahin ulit ang threshold bilang mean + **1.5** SD (kopyahin ang `epidemic_threshold` at palitan ang isang numero). Aling linggo ng 2026 ang unang lumampas dito ngayon? Ilang false alarm ang magiging kapalit nito noong 2022–2025? *(Pahiwatig: i-join ang bawat baseline year sa `thr` sa parehong paraan ng `season` at bilangin ang mga linggong lampas.)*
2. **Parehong linggo lang.** Gamitin lang ang linggong *w* (hindi *w*−1, *w*, *w*+1) ng mga baseline year. Ilang value bawat linggo iyon, at gumagalaw ba ang unang paglampas? *(Pahiwatig: `window_weeks` lang ang kailangang baguhin.)*
3. **Aling sakit ang pinakamabagal?** I-group ang `events` ayon sa `disease` at kuwentahin ang median na `days_to_detect`. Aling sakit ang una mong ituturo sa mga BHW na mapansin?

In [ ]:
# 1. Mas mahigpit na threshold (panimula)
def epidemic_threshold_k(weekly, years, k):
    base = weekly[weekly["iso_year"].isin(years)]
    out = {}
    for week in range(1, 54):
        values = base.loc[base["iso_week"].isin(window_weeks(week)), "cases"]
        out[week] = values.mean() + k * values.std(ddof=1)
    return pd.Series(out)

thr_15 = epidemic_threshold_k(weekly, BASELINE_YEARS, 1.5)
print("Unang linggo ng 2026 na lampas sa mean + 1.5 SD:", (y2026["cases"] > thr_15.loc[y2026.index]).idxmax())

# 3. Pinakamabagal ma-detect na sakit (panimula)
print(events.groupby("disease")["days_to_detect"].median().sort_values(ascending=False).head(3))

## Ang ginawa mo

* Bumuo ng epidemic threshold mula sa **karaniwang** baseline years lang, sinuri ito nang mano-mano, at nahanap ang unang paglampas (2026-W32).
* Gumuhit ng endemic channel at nahuli ang baseline na naglalaman ng epidemya mismo.
* Nakita kung paano pinapababa ng mga huling ulat ang itsura ng pinakabagong mga linggo — basahin ang pinakabagong mga bar bilang "hindi bababa rito".
* Sinukat ang 7-1-7 at ginawang to-do list ang column na `bottleneck`.

Susunod: itatanong ng [kabanata 07](../07_forecast_timesfm.md) kung ano ang maaaring dalhin ng susunod na walong linggo.